# Preliminary exploration for specification search

Preliminary exploration fits a flexible model to training data and uses its
predictions to propose a smaller set of additive-model comparisons. In
`autospec`, the function that proposes a model formula from the data, the preliminary model is a boosted ensemble of depth-4 trees. It
supplies column importance, marginal and partial-dependence shape diagnostics,
and interaction candidates. The final model is fitted from its own formula;
the preliminary ensemble is not a component of that prediction.

The [body-fat case](../cases/autospec_bodyfat.ipynb) demonstrates prediction on
reserved observations. The [controlled case](../cases/preexploration_shapes.ipynb)
checks recovered shapes against a known generating function, and the
[benchmark](../cases/benchmark_preexploration.ipynb) compares the three screening
modes over repeated samples.

## Preliminary ensemble

Let $D=\{(x_i,y_i)\}_{i=1}^n$ be the data supplied to the search and $g(x)$ the
preliminary ensemble. The native engine fits 300 depth-4 squared-error trees (subsampling deterministically to 10,000 rows and capping rounds at 150 when the training table exceeds 10,000 rows)
with learning rate 0.3. Predictors with missing values are filled from these
training rows: a median for numerical columns and a mode for factors. Factors
use level-set splits, so their integer codes do not impose an ordering.
The response is centered and divided by its training standard deviation for
fitting; reported loss increases and interaction variances are converted back
to squared response units.

`preexploration="hessboost"` fits the same preliminary ensemble with a histogram engine, which bins each feature's values before the trees are built, and keeps the same diagnostics. The engine is compiled in only when the build enables the `hessboost` option (a Cargo feature, the Rust build flag); a build without it rejects that choice. `preexploration="off"` skips this ensemble and
retains the full shape catalogue and the residual interaction screen.
With zero budget (the limit on the fits the search may spend), fewer than 20 rows, or insufficient remaining budget, no
preliminary ensemble is fitted. Multinomial and proportional-odds responses
skip this scalar-response exploration.

## Neural and dual preliminary fits

`preexploration="neural"` fits a small, seeded network instead of the tree
ensemble; `"dual"` fits both preliminary models and joins their candidate
reports. The network has two hidden layers of 32 tanh units, a linear output,
and Adam optimization with an internal validation split. Numerical inputs are
standardized from the network's training rows and clipped at five standard
deviations; factors are one-hot encoded. Its work is bounded to 2,000 sampled
rows, 256 encoded inputs and 600 epochs. The best internal-validation epoch
is retained. That validation subset helps fit the *screening model*; it does
not measure adoption of a boosting term.

Both arms use the same partial-dependence and pair-diagnostic vocabulary.
Their `pair_sources` distinguish a tree, neural or combined lead. Agreement
between the two functions is corroborating screening evidence, not an
independent test: they use overlapping input rows and the same candidate
pool, including its eight-column pair limit. The neural arm does not prune
the Gaussian shape catalogue on its own, and neither arm forces a term into
the final fit. A dual probe counts as two native fit equivalents, as defined in Search cost and validation; if the neural arm fails,
its error is reported while the tree proposals remain available. The
[budgeted interaction case](../cases/budgeted_interaction_search.ipynb) follows
a dual proposal through the separate fold comparison. The default is `native`.

## Permutation importance

For evaluation rows sampled from the supplied training data, permute column
$j$ while leaving the other columns fixed. Its importance is

$$I_j=\frac{1}{m}\sum_{i=1}^m
\left[(y_i-g(x_{i,-j},x_{\pi(i),j}))^2-(y_i-g(x_i))^2\right],$$

where $\pi$ is a random permutation. At most 1,000 rows are used. This loss
increase has squared response units and can be negative. The
`importance_fraction` field divides it by the training response variance.
Gaussian shape comparisons follow decreasing importance when budget limits
how many columns can be examined. No column is deleted solely because its
importance is small.

The number measures how much the fitted preliminary model's squared error rises, in-sample, when column $j$ is shuffled. Correlated columns can replace each other, and shuffling can produce combinations that are rare in the training population, so the value is a screening score with those limits. It describes the reliance of the fitted model on column $j$, not a causal effect.

## Partial dependence and marginal diagnostics

Partial dependence averages the preliminary prediction over observed values
of the other predictors. For a single column,

$$g_j(t)=\frac{1}{m}\sum_{i=1}^m g(t,x_{i,-j}).$$

For a pair, $g_{jk}(s,t)$ fixes both columns before the same averaging.
The implementation evaluates at most 100 sampled training rows, computes
single-column curves for at most 32 important columns, and considers pairs
among the eight most important. Columns outside the single-column limit retain
all shape alternatives.

A marginal diagnostic instead compares the response directly with a column.
The Rust `column_screen` routine reports Pearson and Spearman correlations,
linear explained variation, a flexible binned-mean approximation, the best
single split, skewness and the number of distinct values. Its flexible curve
uses at most 20 equal-count bins, with a degrees-of-freedom adjustment:

$$R^2_{\rm curve}=\max\left(0,1-
  \frac{\mathrm{RSS}_{\rm bins}}{\mathrm{TSS}}\frac{n-1}{n-B}\right),$$

when $n>B$, where $B$ is the bin count. Curvature is
$C=\max(0,R^2_{\rm curve}-R^2_{\rm line})$. The step share is
$\max(0,R^2_{\rm step}-R^2_{\rm line})/C$ when $C>0$, and zero otherwise.
The same diagnostics are applied to the preliminary partial-dependence curve.
For an incumbent on a log or square-root scale, the diagnostic abscissa uses
that scale; an invalid transformation domain leaves the alternatives open.

These are heuristic summaries, not tests with calibrated error probabilities.
Binning can hide narrow effects, and partial dependence can evaluate unlikely
combinations of correlated predictors. Ambiguous or weak evidence retains the
full catalogue.

## Shape proposals

The candidate shapes for a numerical column are a line (`bols`, a straight-line term in that column), a smooth (`bbs`, a P-spline: a smooth curve built from a B-spline basis with a penalty on roughness) and a tree (`btree`, a regression tree term; by default one split). Together they form the shape catalogue. For a Gaussian response, pruning first requires permutation importance above
1% of response variance and partial-dependence variance above 0.5%. Numerical
columns then use the following rules, in order:

1. A partial curve with linear $R^2>0.9$, accompanied by marginal linear
   $R^2$ above half the marginal curve $R^2$, proposes a line and a smooth.
2. Partial curvature above 0.1, partial step share above 0.9 and marginal step
   share above 0.8 propose a smooth and a tree.
3. Partial curvature above 0.15, partial step share below 0.65, marginal
   curvature above 0.02 and marginal step share below 0.65 propose a line
   and a smooth.

All other cases retain line, smooth and tree. The incumbent is always included,
and a declared constraint keeps its complete comparison with alternatives.
Other response families can use pair proposals but do not use these
response-scale rules to prune their link-scale shapes.

A proposed replacement still has to improve held-out risk by at least 2%, or
by two paired fold standard errors under the signal-share rule of the shape probe, a held-out comparison of candidate shapes.
The proposal itself cannot replace a term. `rec.evidence["preexploration"]`
records the importance order, diagnostics, proposed shapes and whether shape
pruning applied to the response family.

## Interaction proposals

Friedman and Popescu's H statistic measures departure from additive partial
dependence {cite:p}`friedman2008rules`. Center the three partial-dependence
vectors on their evaluation rows and define

$$H^2_{jk}=\frac{\sum_i[g_{jk}(x_{ij},x_{ik})-g_j(x_{ij})-g_k(x_{ik})]^2}
                 {\sum_i g_{jk}(x_{ij},x_{ik})^2}.$$

The implementation returns zero for a zero denominator and clips the ratio to
$[0,1]$. It also reports the numerator divided by the number of evaluation
rows as an interaction variance, in squared response units. Pair proposals
require H² of at least 0.05 and enough interaction variance to clear the
residual screen's scale-dependent floor. Duplicate-axis checks exclude nearly
identical numerical columns. A large H² alone cannot distinguish a strong
interaction from a negligible joint effect.

The preliminary pair proposals join the residual-screen leads. With
`interactions="adopt"`, the search reserves budget for tuning and held-out
removal comparisons before adding a pair. Removing a retained pair must cost
at least 2% of risk or two paired fold standard errors. Without the required
budget, or after a failed candidate sweep, the pair remains a reported
suggestion and the previous specification is retained.

## Search cost and validation

The preliminary ensemble is counted once and reused for shapes and pairs.
`rec.fits` is the search's reported cost, in full-model fit equivalents (one fit of the full model counts as 1). One-term backfitting comparisons (fits that update a single term against the residuals of the others) count as a fraction of a full-model fit, so `rec.fits` is not a count of all low-level fit calls. Prediction on partial-dependence grids adds work beyond fitting;
that is included when a benchmark times the complete search.

All proposals use the data supplied to `autospec`. Held-out folds protect a
candidate fit's parameter estimation but also participate in adaptive model
search. An inner minimum therefore describes that search and is not an
unbiased generalization estimate. For performance evaluation, screening, tuning and fitting are repeated inside each outer training split, and the resulting model predicts that split's untouched test rows. An engine selected after looking at those test scores needs a fresh
confirmation sample.

## Single-pass fast radar (`search="fast"`)

In exhaustive mode, `autospec` uses preliminary exploration to narrow the candidate set and still runs sequential cross-validated shape, pair, and residual-tree probes. `autospec(..., search="fast")` collapses screening into a single-pass held-out radar (using `sklearn.ensemble.HistGradientBoosting` when available, with a pure-NumPy fallback) that classifies column shapes, weeds near-zero predictors, ranks pairwise interactions, and screens residual spread in one to three sweeps. Candidate additions — interactions, residual trees, and distributional families — are still gated by shared-fold paired standard-error comparisons before adoption, and multi-column tree terms use the Rust histogram-subtraction engine (`split_method="histogram"`).

## Implementation and reproducibility

The numerical pipeline is `mboostrs_core::preexplore::preexplore`; the Python
and R native bindings expose the same report. Python's autospec layer converts
columns and uses the returned proposals. A fixed seed controls row sampling
and permutation. The installed build identity, native-engine feature and
Python source identity should accompany a saved benchmark because the
specification policy also lives in Python.

```{bibliography}
:filter: docname in docnames
```
